# Propriedades

Como classes Pydantic são classes Python comuns, podemos adicionar métodos e propriedades.

Por exemplo:

In [1]:
from pydantic import BaseModel, Field, ValidationError

In [2]:
from math import pi


class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0)

    def area(self):
        return pi * self.radius ** 2

Criamos a instância normalmente:

In [3]:
c = Circle(center=(1, 1), radius=2)

Ao serializar, só aparecem os campos:

In [4]:
print(f"c.model_dump(): {c.model_dump()!r}")

c.model_dump(): {'center': (1, 1), 'radius': 2}


Mas a instância tem o método `area()`:

In [5]:
print(f"c.area(): {c.area()!r}")

c.area(): 12.566370614359172


Também podemos adicionar propriedades — usemos uma propriedade em vez de método para a área:

In [6]:
class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0)

    @property
    def area(self):
        return pi * self.radius ** 2

Funciona como antes:

In [7]:
m = Circle()

In [8]:
print(f"m.area: {m.area!r}")

m.area: 3.141592653589793


Mas a propriedade não aparece na representação nem na serialização:

In [9]:
print(f"m: {m!r}")

m: Circle(center=(0, 0), radius=1)


In [10]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'center': (0, 0), 'radius': 1}


In [11]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"center":[0,0],"radius":1}'


E, inspecionando os campos do modelo (pela classe — acessar `model_fields` pela instância está depreciado desde a 2.11), `area` também não aparece:

In [12]:
print(f"Circle.model_fields: {Circle.model_fields!r}")

Circle.model_fields: {'center': FieldInfo(annotation=tuple[int, int], required=False, default=(0, 0)), 'radius': FieldInfo(annotation=int, required=False, default=1, metadata=[Gt(gt=0)])}


Então, embora possamos adicionar propriedades a modelos Pydantic, elas não são serializadas. É preciso um pouco mais de trabalho para incluí-las.

Antes disso: também podemos **cachear** propriedades.

Neste exemplo poderíamos cachear a `area` — mas aí precisamos garantir que o campo `radius` seja imutável (congelado). Poderíamos congelar o
círculo inteiro ou só `radius`.

Vamos congelar só `radius`.

In [13]:
from functools import cached_property

class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int = Field(default=1, gt=0, frozen=True)

    @cached_property
    def area(self):
        print("calculating area...")
        return pi * self.radius ** 2

In [14]:
c = Circle()

In [15]:
print(f"c.area: {c.area!r}")

calculating area...
c.area: 3.141592653589793


O código da propriedade rodou; mas, acessando de novo:

In [16]:
print(f"c.area: {c.area!r}")

c.area: 3.141592653589793


a implementação não roda outra vez.

E está tudo certo, porque `radius` agora é congelado:

In [17]:
try:
    c.radius=2
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Circle
radius
  Field is frozen [type=frozen_field, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/frozen_field


Ou seja: podemos adicionar qualquer método, propriedade e propriedade cacheada às classes Pydantic — mas propriedades não viram campos do modelo.